### Phase 1

In [3]:
import numpy as np
import copy

#phase1:setup
#classes, deck setup,turns and states defined

class Card:
    #initializes a card object with a specific color and value
    def __init__(self, color, value):
        self.color = color
        self.value = value
    
    def __repr__(self):
        return f"{self.color} {self.value}"
    #allows direct comparison bw two card objects based on their color and value   
    def __eq__(self, other):
        if isinstance(other, Card):
            return self.color == other.color and self.value == other.value
        return False

def generate_deck():
    colors = ['Red', 'Blue', 'Green', 'Yellow'] 
    values = [str(i) for i in range(10)] + ['Skip'] 
    
    deck_list = []
    for c in colors:
        for v in values:
            new_card = Card(c, v)
            deck_list.append(new_card)
            
    deck = np.array(deck_list)
    np.random.shuffle(deck)
    return deck.tolist()

def initialize_game():
    deck = generate_deck()
    state = {
        'p1_cards': [deck.pop() for _ in range(5)], 
        'p2_cards': [deck.pop() for _ in range(5)], 
        'p3_cards': [deck.pop() for _ in range(5)], 
        'top_card': deck.pop(), 
        'deck': deck,
        'current_turn': 1 
    }
    return state

def get_valid_moves(hand, top_card):
    valid_moves = []
    for c in hand:
        if c.color == top_card.color or c.value == top_card.value:
            valid_moves.append(c)
    return valid_moves

def apply_move(state, player_key, move):
    #Deepcopies the state, processes the card play or draw, and advances the turn (handling Skip logic)
    new_state = copy.deepcopy(state)
    
    if move == 'Draw':
        if new_state['deck']:
            new_state[player_key].append(new_state['deck'].pop())
        new_state['current_turn'] = (new_state['current_turn'] % 3) + 1
    else:
        new_state[player_key].remove(move)
        new_state['top_card'] = move
        if move.value == 'Skip':
            new_state['current_turn'] = (new_state['current_turn'] + 1) % 3 + 1
        else:
            new_state['current_turn'] = (new_state['current_turn'] % 3) + 1
            
    return new_state

def is_terminal(state):
    if len(state['p1_cards']) == 0:
        return True
    elif len(state['p2_cards']) == 0:
        return True
    elif len(state['p3_cards']) == 0:
        return True
    else:
        return False

In [4]:
#partial testing of phase 1
test_state = initialize_game()

print("initial states")
print("Top Card:", test_state['top_card'])
print("Player 1 Hand:", test_state['p1_cards'])
print("Player 2 Hand:", test_state['p2_cards'])
print("Player 3 Hand:", test_state['p3_cards'])

#testing valid moves
p1_hand = test_state['p1_cards']
current_top = test_state['top_card']
p1_moves = get_valid_moves(p1_hand, current_top)
print("\nPlayer 1 Valid Moves:", p1_moves)

if p1_moves:
    chosen_move = p1_moves[0]
else:
    chosen_move = 'Draw'

print("Applying Move:", chosen_move)
new_state = apply_move(test_state, 'p1_cards', chosen_move)
print("\nNew Game State")
print("Turn:", new_state['current_turn'])
print("New Top Card:", new_state['top_card'])
print("Player 1 New Hand:", new_state['p1_cards'])

initial states
Top Card: Red 3
Player 1 Hand: [Red Skip, Green 9, Green 1, Red 7, Blue 5]
Player 2 Hand: [Green 3, Blue 3, Red 1, Green 4, Green 5]
Player 3 Hand: [Red 0, Green 2, Yellow 6, Red 8, Yellow 4]

Player 1 Valid Moves: [Red Skip, Red 7]
Applying Move: Red Skip

New Game State
Turn: 3
New Top Card: Red Skip
Player 1 New Hand: [Green 9, Green 1, Red 7, Blue 5]


### Phase 2

In [5]:
#phase 2: eval function
#adjusts weights etc here
#also drew handwritten part here as well as the explanation of functions in markdown

def get_opponent_keys(player_key):
    players = ['p1_cards', 'p2_cards', 'p3_cards']
    opponents = []
    for p in players:
        if p != player_key:
            opponents.append(p)
    return opponents

def base_evaluation(state, player_key, weights):
    ai_hand = state[player_key]
    opp_keys = get_opponent_keys(player_key)
    c_ai = len(ai_hand)
    
    #calculates average opponent cards
    opp1_key = opp_keys[0]
    opp2_key = opp_keys[1]
    opp1_count = len(state[opp1_key])
    opp2_count = len(state[opp2_key])
    c_opp = (opp1_count + opp2_count) / 2
    s_cards = 0
    for card in ai_hand:
        if card.value == 'Skip':
            s_cards = s_cards + 1

    # score = 50 + (w1 * c_ai) + (w2 * c_opp) + (w3 * s_cards)
    score = 50
    score = score + (weights[0] * c_ai)
    score = score + (weights[1] * c_opp)
    score = score + (weights[2] * s_cards)
    
    return score

def eval_defensive(state, player_key):
    weights = [-5, 2, 3]
    score = base_evaluation(state, player_key, weights)
    return score

def eval_offensive(state, player_key):
    weights = [-8, 1, 1]
    score = base_evaluation(state, player_key, weights)
    return score

In [6]:
#partial testing of phase 2
test_state = initialize_game()

print("hands for eval")
print("Player 1 Hand (Focus AI):", test_state['p1_cards'])
print("Player 2 Hand (Opponent):", test_state['p2_cards'])
print("Player 3 Hand (Opponent):", test_state['p3_cards'])

#counts skips for P1 manually
p1_hand = test_state['p1_cards']
p1_skips = 0
for card in p1_hand:
    if card.value == 'Skip':
        p1_skips = p1_skips + 1

print("\nPlayer 1 currently holds", p1_skips, "'Skip' cards.")
defensive_score = eval_defensive(test_state, 'p1_cards')
offensive_score = eval_offensive(test_state, 'p1_cards')
print("\ncalculated AI Scores for Player 1")
print("Defensive Score [-5, 2, 3]:", defensive_score)
print("Offensive Score [-8, 1, 1]:", offensive_score)

hands for eval
Player 1 Hand (Focus AI): [Blue 1, Blue 3, Red 4, Green 9, Green 4]
Player 2 Hand (Opponent): [Red 5, Green 8, Green Skip, Red 9, Red 1]
Player 3 Hand (Opponent): [Yellow 1, Blue 8, Green 5, Yellow 9, Red 0]

Player 1 currently holds 0 'Skip' cards.

calculated AI Scores for Player 1
Defensive Score [-5, 2, 3]: 35.0
Offensive Score [-8, 1, 1]: 15.0


### Evaluation Functions Explanation

The AI uses a simple formula to score how "good" its current hand is. A higher score means the AI is in a better position to win. 

Formula: Score = 50 - (AI Cards * W1) + (Opponent Cards * W2) + (Skip Cards * W3)

tuned the weights (W) differently for two play styles:

1. Defensive Mode (Weights: -5, 2, 3):** Focuses on survival. It highly values hoarding "Skip" cards to block opponents and prefers opponents to have lots of cards.
2. Offensive Mode (Weights: -8, 1, 1):** Focuses on attacking. It heavily penalizes holding *any* cards, forcing the AI to play its hand aggressively and reach zero cards as fast as possible.

### Phase 3

In [7]:
#phase 3: minimax and expectimax algos
#also adjusted handwritten tree while this phase that i started in phase 2
def minimax(state, depth, maximizing_player, player_key):
    is_done = is_terminal(state)
    if depth == 0 or is_done:
        score = eval_defensive(state, player_key)
        return score, None

    valid_moves = get_valid_moves(state[player_key], state['top_card'])
    if not valid_moves:
        valid_moves = ['Draw']

    if maximizing_player:
        max_eval = float('-inf')
        best_move = valid_moves[0]
        for move in valid_moves:
            new_state = apply_move(state, player_key, move)
            results = minimax(new_state, depth - 1, False, player_key)
            eval_score = results[0]
            
            if eval_score > max_eval:
                max_eval = eval_score
                best_move = move
        return max_eval, best_move
        
    else:
        min_eval = float('inf')
        best_move = valid_moves[0]
        for move in valid_moves:
            new_state = apply_move(state, player_key, move)
            results = minimax(new_state, depth - 1, True, player_key)
            eval_score = results[0]
            
            if eval_score < min_eval:
                min_eval = eval_score
                best_move = move
        return min_eval, best_move

def expectimax(state, depth, is_ai_turn, player_key):
    is_done = is_terminal(state)
    if depth == 0 or is_done:
        score = eval_offensive(state, player_key)
        return score, None

    valid_moves = get_valid_moves(state[player_key], state['top_card'])

    if is_ai_turn: 
        #when ai has to draw card
        if not valid_moves:
            expected_value = 0
            deck_cards = state['deck']
            if deck_cards:
                deck_length = len(deck_cards)
                prob = 1 / deck_length
                for card in deck_cards:
                    new_state = apply_move(state, player_key, 'Draw')
                    #here we set drawn card specifically
                    new_state[player_key][-1] = card 
                    results = expectimax(new_state, depth - 1, False, player_key)
                    eval_score = results[0]
                    expected_value = expected_value + (prob * eval_score)
            return expected_value, 'Draw'
            
        #maximizing logic
        max_eval = float('-inf')
        best_move = valid_moves[0]
        for move in valid_moves:
            new_state = apply_move(state, player_key, move)
            results = expectimax(new_state, depth - 1, False, player_key)
            eval_score = results[0]
            if eval_score > max_eval:
                max_eval = eval_score
                best_move = move
        return max_eval, best_move
        
    else:
        if valid_moves:
            move = np.random.choice(valid_moves)
        else:
            move = 'Draw'
            
        new_state = apply_move(state, player_key, move)
        results = expectimax(new_state, depth - 1, True, player_key)
        eval_score = results[0]
        return eval_score, move

In [8]:
test_state = initialize_game()
print("current top card and cards that AI has")
top_card = test_state['top_card']
p1_hand = test_state['p1_cards']
p2_hand = test_state['p2_cards']

print("top Card:", top_card)
print("P1 Hand (Def):", p1_hand)
print("P2 Hand (Off):", p2_hand)
print("\nTesting P1 Minimax (Depth 2)")

#minimax
p1_results = minimax(test_state, 2, True, 'p1_cards')
p1_score = p1_results[0]
p1_move = p1_results[1]
print("Minimax chooses:", p1_move)
print("Expected Score:", p1_score)

print("\ntesting P2 Expectimax (Depth 2)")
#expectimax
p2_results = expectimax(test_state, 2, True, 'p2_cards')
p2_score = p2_results[0]
p2_move = p2_results[1]
print("Expectimax chooses:", p2_move)
print("Expected Score:", p2_score)

current top card and cards that AI has
top Card: Blue 1
P1 Hand (Def): [Blue 4, Green 0, Blue 0, Green 7, Red 4]
P2 Hand (Off): [Red 0, Blue 3, Green 1, Yellow 6, Yellow 4]

Testing P1 Minimax (Depth 2)
Minimax chooses: Blue 4
Expected Score: 45.0

testing P2 Expectimax (Depth 2)
Expectimax chooses: Blue 3
Expected Score: 15.0
